# NVIDIA Nemotron — EDA
Explore the 9,500 training puzzles.

In [ ]:
import csv
import json
from collections import Counter
import re

with open('../data/train.csv') as f:
    rows = list(csv.DictReader(f))

print(f'Total training examples: {len(rows)}')
print(f'Columns: {list(rows[0].keys())}')

In [ ]:
# Classify puzzle types
def classify(prompt):
    p = prompt.lower()
    if 'bit manipulation' in p or '8-bit binary' in p:
        return 'bit_manipulation'
    elif 'gravitational constant' in p:
        return 'physics_constant'
    elif 'unit conversion' in p:
        return 'unit_conversion'
    elif 'encryption' in p or 'cipher' in p or 'encrypt' in p:
        return 'text_encryption'
    elif 'numbers are secretly converted' in p or 'number system' in p:
        return 'number_system'
    elif 'algebraic' in p or 'equation' in p:
        return 'algebraic'
    return 'other'

for r in rows:
    r['type'] = classify(r['prompt'])

counts = Counter(r['type'] for r in rows)
for t, c in counts.most_common():
    print(f'{t:25s}: {c:5d} ({100*c/len(rows):.1f}%)')

In [ ]:
# Show one example per type
seen = set()
for r in rows:
    if r['type'] not in seen:
        seen.add(r['type'])
        print(f"=== {r['type']} ===")
        print('PROMPT:', r['prompt'][:400])
        print('ANSWER:', r['answer'])
        print()

In [ ]:
# Answer format analysis
def answer_format(ans):
    if re.fullmatch(r'[01]{8}', ans):
        return 'binary_8bit'
    elif re.fullmatch(r'[IVXLCDM]+', ans):
        return 'roman_numeral'
    elif re.fullmatch(r'-?\d+\.\d+', ans):
        return 'decimal'
    elif re.fullmatch(r'-?\d+', ans):
        return 'integer'
    elif re.fullmatch(r'[a-z ]+', ans):
        return 'lowercase_text'
    else:
        return 'other'

fmt_counts = Counter(answer_format(r['answer']) for r in rows)
for fmt, c in fmt_counts.most_common():
    print(f'{fmt:20s}: {c}')

In [ ]:
# Prompt length distribution
lengths = [len(r['prompt']) for r in rows]
print(f'Prompt length — min: {min(lengths)}, max: {max(lengths)}, avg: {sum(lengths)//len(lengths)}')

# Check test set
with open('../data/test.csv') as f:
    test_rows = list(csv.DictReader(f))
print(f'\nTest examples: {len(test_rows)}')
for r in test_rows:
    print('ID:', r['id'], '| type:', classify(r['prompt']))
    print('PROMPT:', r['prompt'][:200])
    print()